# GRU Neural Network for Insurance Claims Reserving

This notebook implements a Gated Recurrent Unit (GRU) neural network for predicting ultimate insurance claim amounts. The model processes sequential payment data to learn temporal patterns in claim development. This version processes data in a triangular format, with "3D" input of claims history to predict "2D" output of future claims history. 

## Overview
- **Objective**: Predict ultimate claim amounts using payment history
- **Data**: 
   * **X**: Entire claims history with features summarised by month, as padded sequence. 
   * **y**: all future payments, multi-output by month 
- **Model**: GRU with log-link activation for positive predictions
- **Features**: Temporal claim development data (occurrence time, development period, payment amounts, etc.). More features are available in source data than are used, but kept consistent with other notebooks for comparability (e.g. no outstanding).
- **Evaluation**: Actual vs Expected plots, QQ plots, and model interpretability via SHAP

## Import Libraries

We import essential libraries for data processing, neural networks, and visualization.

In addition we import a number of custom functions from the `utils` folder, developed for this notebook but stored separately to simplify this flow and presentation.


In [1]:
import pandas as pd
import numpy as np

from matplotlib import pyplot as plt
import time
from datetime import datetime

# PyTorch imports

from torch.utils.tensorboard import SummaryWriter

# Scikit-learn imports
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import MinMaxScaler
from sklearn.pipeline import Pipeline


# Local imports
from utils.config import ExperimentConfig, load_config_from_yaml
from utils.neural_networks import TabularNetRegressor, BasicLogGRU, ColumnKeeper, Make3D

from utils.data_engineering import load_data, process_data_grouped_triangular, create_train_test_datasets_seq_3D, make_claim_sampler
from utils.tensorboard import generate_enhanced_tensorboard_outputs, create_actual_vs_expected_plot

from utils.excel import save_df_to_excel

/Users/jacky/GitRepos/MLR_working_party/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Configuration Setup

We configure the experiment settings, including model parameters, training settings, and output options.

Our configuration is read a config file in the utils folder.

In [2]:
# Load from YAML file
config = load_config_from_yaml('configs/GRU_triangular_config.yaml')

# Set pandas display options
pd.options.display.float_format = '{:,.2f}'.format

# Enable inline plotting for Jupyter notebooks
%matplotlib inline

SEED =  config['training'].seed # 42 
rng = np.random.default_rng(SEED) 

# Create timestamp for logging
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
run_name = f"GRU_experiment_triangular_{timestamp}"  # Customize as needed

# Initialize TensorBoard writer
writer = SummaryWriter(log_dir=f"runs/{run_name}")
log_filename = f"logs/{run_name}.xlsx"

print(f"Experiment timestamp: {timestamp}")
print(f"Output file: {log_filename}")

# Save excel data. Set this to True to better understand the data structure.
save_excel_data = False

Experiment timestamp: 20260303_221824
Output file: logs/GRU_experiment_triangular_20260303_221824.xlsx


## Data Loading and Processing

The dataset used is SPLICE (Synthetic Paid Loss and Incurred Cost Experience), which is based on the R data simulation package SynthETIC ([Synthetic](https://cran.rstudio.com/web/packages/SynthETIC/index.html)). This is an interesting solution because is possible generate claims with some specific features.
Here a full description of the mentioned R package ([SynthETIC intro](https://institute-and-faculty-of-actuaries.github.io/mlr-blog/post/data/synthetic/)).

We load the raw insurance claims data and process it for neural network training. This includes:
- Loading raw data from CSV
- Feature engineering (development periods, payment indicators, etc.)
- Creating train/test splits based on date cutoff

In [3]:
# Load original data
dat_orig = load_data(config)
print(f"Original data shape: {dat_orig.shape}")
print(f"Original data columns: {list(dat_orig.columns)}")

# Save original data to Excel
if save_excel_data:
    save_df_to_excel(dat_orig, df_name="Original Data", filename=log_filename, mode='w')
dat_orig.head()

Original data shape: (19322, 13)
Original data columns: ['Unnamed: 0', 'claim_no', 'pmt_no', 'occurrence_period', 'occurrence_time', 'claim_size', 'notidel', 'setldel', 'payment_time', 'payment_period', 'payment_size', 'payment_inflated', 'payment_delay']


,Unnamed: 0,claim_no,pmt_no,occurrence_period,occurrence_time,claim_size,notidel,setldel,payment_time,payment_period,payment_size,payment_inflated,payment_delay
0,1,1,1,1,0.73,"232,310.09",0.66,23.21,5.33,6,"13,226.34","13,226.34",3.93
1,2,1,2,1,0.73,"232,310.09",0.66,23.21,10.09,11,"15,685.86","15,685.86",4.76
2,3,1,3,1,0.73,"232,310.09",0.66,23.21,18.02,19,"14,643.28","14,643.28",7.93
3,4,1,4,1,0.73,"232,310.09",0.66,23.21,22.82,23,"170,041.89","170,041.89",4.79
4,5,1,5,1,0.73,"232,310.09",0.66,23.21,24.61,25,"18,712.71","18,712.71",1.79




Processing code for this dataset (credit Davide, Nigel) has been put into the function 'process_data_grouped_triangular' part of the utils functions. It's not a particularly elegant solution but represent a half-way house towards standardised data processing functions. 

The function processes the input dataset to generate 40 development periods, 40 occurrence periods and around 3,600 claims. The data is converted to a time series with one record per time period (even if multiple or no claims transactions). Inspired from the MLWP tutorial here: http://institute-and-faculty-of-actuaries.github.io/mlr-blog/post/research/chain_ladder_to_individual_mdn/#introduction

We should note that, in a departure from the original notebook, incurred and outstanding data features are not used in this analysis. This is a deliberate choice to help compare these outputs to Sarah's work using GRUs that used paid datasets only. As it turns out, Sarah's GRU approach is deliberately structurally different and so the use of paid data only in this notebook is not sufficient to enable the direct comparison of approaches. 

This is likely too much detail for now. However the interested reader may observe that omission of incurred and outstanding features do result in slightly poorer model performance which can be seen by comparing QQ plots and oher exhibits from this notebook to the original. 


In [4]:
# Process data (feature engineering, filtering, etc.)
dat = process_data_grouped_triangular(config, dat_orig)
print(f"Processed data shape: {dat.shape}")
print(f"New columns added: {set(dat.columns) - set(dat_orig.columns)}")

# Save processed data to Excel
if save_excel_data:
    save_df_to_excel(dat, df_name="Processed Data", filename=log_filename, mode='a')

Processed data shape: (142841, 25)
New columns added: {'cv_ind', 'payment_count_to_prior_period', 'log1_payment_to_prior_period', 'occurrence_date', 'payment_to_prior_period', 'log1_cumulative_payment_to_prior_period', 'has_payment_to_prior_period', 'backdate_periods', 'is_settled', 'train_ind', 'payment_date', 'payment_period_as_at', 'data_as_at_development_period', 'payment_size_cumulative'}


/Users/jacky/GitRepos/MLR_working_party/02_code/utils/data_engineering.py:198: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  dat['occurrence_date'] = pd.to_datetime(dat['occurrence_date']).dt.to_period('Q')
/Users/jacky/GitRepos/MLR_working_party/02_code/utils/data_engineering.py:199: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  dat['payment_date'] = pd.to_datetime(dat['payment_date']).dt.to_period('Q')


## Train/Test Split Creation

We create training and test datasets:
- **Training**: Transactions before the cutoff date
- **Testing**: All transactions including transactions after the cutoff date

In [5]:
# Create train/test datasets
w_train, x_train, y_train, w_test, x_test, y_test = create_train_test_datasets_seq_3D(dat, config)

print(f"Training features shape: {x_train.shape}")
print(f"Training targets shape: {y_train.shape}")
print(f"Test features shape: {x_test.shape}")
print(f"Test targets shape: {y_test.shape}")

# Save datasets to Excel
if save_excel_data:
    save_df_to_excel(w_train, df_name="w_train", filename=log_filename, mode='a')
    save_df_to_excel(x_train, df_name="x_train", filename=log_filename, mode='a')
    save_df_to_excel(y_train, df_name="y_train", filename=log_filename, mode='a')

    save_df_to_excel(w_test, df_name="w_test", filename=log_filename, mode='a')
    save_df_to_excel(x_test, df_name="x_test", filename=log_filename, mode='a')
    save_df_to_excel(y_test, df_name="y_test", filename=log_filename, mode='a')

Training features shape: (72757, 8)
Training targets shape: (72757, 41)
Test features shape: (142841, 8)
Test targets shape: (142841, 41)


In [6]:
# Extract configuration values for model setup
features = config['data'].features
data_cols = config['data'].data_cols
youtput = config['data'].output_field

# Print dataset info
nclms = x_train['claim_no'].nunique()
nfeatures = len(features)

print(f"Number of unique claims in training: {nclms}")
print(f"Number of features: {nfeatures}")
print(f"Features: {features}")
print(f"Target variable: {youtput}")

Number of unique claims in training: 3571
Number of features: 7
Features: ['notidel', 'occurrence_time', 'development_period', 'payment_period', 'has_payment_to_prior_period', 'log1_payment_to_prior_period', 'payment_count_to_prior_period']
Target variable: claim_size


## Model Pipeline Setup


To manage the significantly expanded dataset resulting from our "backdating" augmentation, we implement a dynamic sampling strategy for model training. This involves a custom claim_sampler function, which, for each training epoch or batch, randomly selects just one historical snapshot for every unique (claim_no, development_period) combination. This approach efficiently utilizes the rich, augmented data by exposing the model to diverse temporal perspectives, thereby enhancing generalization and optimizing computational resources during the learning process.

In [7]:
# Sampling to reduce increase in batch size due to data augmentation
claim_sampler = make_claim_sampler(y_train)

In [8]:
y_train

,claim_no,development_period,future_payment_1,future_payment_2,future_payment_3,future_payment_4,future_payment_5,future_payment_6,future_payment_7,future_payment_8,...,future_payment_30,future_payment_31,future_payment_32,future_payment_33,future_payment_34,future_payment_35,future_payment_36,future_payment_37,future_payment_38,future_payment_39
1,1,1,0.00,0.00,0.00,"13,226.34",0.00,0.00,0.00,0.00,...,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
2,1,2,0.00,0.00,"13,226.34",0.00,0.00,0.00,0.00,"15,685.86",...,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
3,1,3,0.00,"13,226.34",0.00,0.00,0.00,0.00,"15,685.86",0.00,...,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
4,1,4,"13,226.34",0.00,0.00,0.00,0.00,"15,685.86",0.00,0.00,...,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
5,1,5,0.00,0.00,0.00,0.00,"15,685.86",0.00,0.00,0.00,...,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
146160,3655,0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,...,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
146240,3657,0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,...,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
146320,3659,0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,...,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
146360,3660,0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,...,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00



We create a scikit-learn pipeline that:
1. **ColumnKeeper**: Selects relevant columns
2. **MinMaxScaler**: Normalizes features to [0,1] range (critical for neural networks)
3. **Make3D**: Converts tabular data to 3D tensors for GRU processing
4. **TabularNetRegressor**: Wraps the GRU model with training logic

In [9]:
# Create preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        # Scale 0-1, but preserve development_period for the logic
        ('scale', MinMaxScaler(), 
         [x for x in features if x != "development_period"])],
    remainder='passthrough',
    verbose_feature_names_out=False
)

preprocessor.set_output(transform="pandas")

model_NN = Pipeline(
    steps=[
        ("keep", ColumnKeeper(data_cols)),   
        ('zero_to_one', preprocessor),       # Important! Standardize deep learning inputs.
        ('3Dtensor', Make3D(features, full_history=True, config=config)),
        ("model", TabularNetRegressor(
            BasicLogGRU, 
            n_hidden=config['model'].n_hidden, 
            max_iter=config['training'].nn_iter,
            batch_function=claim_sampler if config['training'].use_batching_logic else None,                
            rebatch_every_iter=config['training'].rebatch_every_iter,  # takes time to resample so iterate a few epochs per resample,            
            enable_shap=config['training'].enable_shap,
            shap_log_frequency=config['tensorboard'].shap_log_frequency,
            seed=SEED,
            config=config,
            device="default"
        ))
    ]
)

print("Model pipeline created successfully")
print(f"Hidden units: {config['model'].n_hidden}")
print(f"Training iterations: {config['training'].nn_iter}")

Model pipeline created successfully
Hidden units: 32
Training iterations: 2001


In [ ]:
# Model initial parameters
model_NN.named_steps["model"]

,module,<class 'utils....BasicLogGRU'>
,criterion,<class 'torch...issonNLLLoss'>
,max_iter,2001
,max_lr,0.01
,keep_best_model,False
,batch_function,<function mak...t 0x17dd5d8a0>
,rebatch_every_iter,100
,n_hidden,32
,l1_penalty,0.0
,l1_applies_params,"['linear.weight', 'hidden.weight']"
,weight_decay,0.0


## Model Training

We train the GRU model on the sequential payment data. The model learns to predict ultimate claim amounts from partial payment histories.



In [11]:
def train_model(model, x_train, y_train, config: ExperimentConfig):
    """
    Train the neural network model with timing.
    
    Args:
        model: The model pipeline to train
        x_train: Training features
        y_train: Training targets
        config: Experiment configuration
        
    Returns:
        Trained model and elapsed time
    """
    print("Starting model training...")
    start_time = time.time()
    
    model.fit(x_train, y_train)
    
    end_time = time.time()
    elapsed_time = end_time - start_time
    print(f"Training completed. Execution time: {elapsed_time:.2f} seconds")
    
    return model, elapsed_time
  
# Train the model
trained_model, training_time = train_model(model_NN, x_train, y_train, config)

Starting model training...
torch.Size([72757, 40, 7])
SHAP explainer initialized successfully
Epoch: 0 Train RMSE: 26026.205078125 Train Loss: -11856.1826171875
refreshing batch on epoch 0
refreshing batch on epoch 100
Epoch: 200 Train RMSE: 25971.15625 Train Loss: -12428.4033203125
refreshing batch on epoch 200
refreshing batch on epoch 300
Epoch: 400 Train RMSE: 25955.77734375 Train Loss: -12467.033203125
refreshing batch on epoch 400
refreshing batch on epoch 500
Epoch: 600 Train RMSE: 25876.224609375 Train Loss: -12986.21875
refreshing batch on epoch 600
refreshing batch on epoch 700
Epoch: 800 Train RMSE: 25960.08203125 Train Loss: -12571.298828125
refreshing batch on epoch 800
refreshing batch on epoch 900
Epoch: 1000 Train RMSE: 26027.41015625 Train Loss: -11903.767578125
refreshing batch on epoch 1000
refreshing batch on epoch 1100
Epoch: 1200 Train RMSE: 26010.4140625 Train Loss: -12074.490234375
refreshing batch on epoch 1200
refreshing batch on epoch 1300
Epoch: 1400 Train R

## Training Results and SHAP Analysis

After training, we generate predictions and SHAP explanations to understand feature importance. SHAP values show how each feature contributes to individual predictions.

To observe the tensorboard logs, run the following command in your terminal:

`tensorboard --logdir=./02_code/runs/`

In [12]:
train = (dat.loc[(dat.train_ind_time == 1) & (dat.train_ind == 1) & (dat.train_settled == 1)])

# Generate enhanced outputs including SHAP analysis
train_pred = generate_enhanced_tensorboard_outputs(trained_model, train, config, writer)
save_df_to_excel(train_pred, df_name="pred_train", filename=log_filename, mode='a')


# Generate basic predictions
y_pred = trained_model.predict(x_train)
save_df_to_excel(pd.DataFrame(y_pred, columns=['prediction']), df_name="y_pred_train", filename=log_filename, mode='a')

print(f"Training predictions generated for {len(train_pred)} records")
print(f"Mean predicted claim amount: ${train_pred['pred_claims'].mean():,.2f}")
print(f"Mean actual claim amount: ${train_pred[youtput].mean():,.2f}")

AttributeError: 'DataFrame' object has no attribute 'train_ind_time'

## Training Set Evaluation Plots

We create several diagnostic plots to evaluate model performance on training data:
- **Actual vs Expected**: Scatter plots comparing predictions to actual values
- **Logged Plots**: Log-transformed versions for better visualization of large values
- **Ultimate Claims**: Predictions for final claim amounts only

In [ ]:

# Actual vs Expected - All Training History
fig, ax = create_actual_vs_expected_plot(
    train_pred, youtput, "pred_claims", 
    'Train AvsE all history', 
    writer, 'AvsE all Train'
)
fig

In [ ]:
# Logged Actual vs Expected - Training
fig, ax = create_actual_vs_expected_plot(
    train_pred, "log_actual", "log_pred_claims", 
    'Logged Train AvsE All History', 
    writer, 'AvsE Logged All Train',
    max_val=20
)

fig

In [ ]:
# AvsE Ult - Train
dat_byclaim = train_pred.groupby("claim_no").last()
fig, ax = create_actual_vs_expected_plot(
    dat_byclaim, "claim_size", "pred_claims", 
    'Train AvsE Ult only', 
    writer, 'AvsE Ult only Train'
)

fig

We create several QQ diagnostic plots to evaluate model performance on training data:
- **Actual vs Expected**: Scatter plots comparing predictions to actual values
- **Logged Plots**: Log-transformed versions for better visualization of large values

In [ ]:
train_pred["pred_claims_20cile"] = pd.qcut(train_pred["pred_claims"], 20, labels=False, duplicates='drop')
#X_sum = train_pred.groupby("pred_claims_20cile").agg("mean").reset_index()
#X_sum = train_pred.groupby("pred_claims_20cile")[["payment_size", "pred_claims", "log_actual", "log_pred_claims"]].mean().reset_index()
X_sum = train_pred.groupby("pred_claims_20cile").mean(numeric_only=True).reset_index()

#X_sum = train_pred.groupby("pred_claims_20cile").agg(
#    payment_size_mean=("payment_size","mean"),
#    pred_claims_mean=("pred_claims","mean")
#).reset_index()

# QQ - Train
fig, ax  = create_actual_vs_expected_plot(
    X_sum, "payment_size", "pred_claims", 
    'Train QQ plot 20', 
    writer, 'QQ plot Train'
)
fig


In [ ]:

# Logged QQ - Train
fig, ax  = create_actual_vs_expected_plot(
    X_sum, "log_actual", "log_pred_claims", 
    'Logged Train QQ plot 20', 
    writer, 'QQ plot Logged Train',
    max_val = 20,
)

fig

## Test Set Evaluation Plots

We then repeat the process to create several diagnostic plots to evaluate model performance on test data:
- **Actual vs Expected**: Scatter plots comparing predictions to actual values
- **Logged Plots**: Log-transformed versions for better visualization of large values
- **Ultimate Claims**: Predictions for final claim amounts only

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Create plots TEST
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

test = (dat.loc[(dat.test_ind_time == 1) & (dat.train_ind == 0) & (dat.train_settled == 0)])

youtput="claim_size"
y_pred=model_NN.predict(test)

y_predx=model_NN.predict(x_test)

#merge y_pred back into dat for each claim
claim_nos = test["claim_no"].drop_duplicates()
pred_df = pd.DataFrame({
    "claim_no": claim_nos.values,
    "pred_claims": y_pred
})

if "pred_claims" in test.columns:
    dat = dat.drop(columns=["pred_claims"])

test_pred = test.merge(pred_df, on="claim_no", how="left")

test_pred["log_pred_claims"]=test_pred["pred_claims"].apply(lambda x: np.log(x+1))
test_pred["log_actual"]=test_pred[youtput].apply(lambda x: np.log(x+1))

test_pred["rpt_delay"]=np.ceil(train_pred.notidel).astype(int)

test_pred["diff"]=test_pred[youtput]-train_pred["pred_claims"]
test_pred["diffp"]=(test_pred[youtput]-test_pred["pred_claims"])/test_pred[youtput]

save_df_to_excel(test_pred, df_name="pred_test", filename=log_filename, mode='a')


In [ ]:
# AvsE all - Test
fig, ax = create_actual_vs_expected_plot(
    test_pred, youtput, "pred_claims", 
    'Test AvsE all history', 
    writer, 'AvsE all Test'
)

fig

In [ ]:
# Logged AvsE all - Test
fig, ax = create_actual_vs_expected_plot(
    test_pred, "log_actual", "log_pred_claims", 
    'Logged Test AvsE All History', 
    writer, 'AvsE Logged All Test',
    max_val=20
)

fig

In [ ]:
# AvsE Ult - Test
dat_byclaim = test_pred.groupby("claim_no").last()
fig, ax = create_actual_vs_expected_plot(
    dat_byclaim, "claim_size", "pred_claims", 
    'Test AvsE Ult only', 
    writer, 'AvsE Ult only Test'
)

fig

We create several QQ diagnostic plots to evaluate model performance on test data:
- **Actual vs Expected**: Scatter plots comparing predictions to actual values
- **Logged Plots**: Log-transformed versions for better visualization of large values

In [ ]:
test_pred["pred_claims_20cile"] = pd.qcut(test_pred["pred_claims"], 20, labels=False, duplicates='drop')
#X_sum = test_pred.groupby("pred_claims_20cile").agg("mean").reset_index()
X_sum = test_pred.groupby("pred_claims_20cile").mean(numeric_only=True).reset_index()

# QQ - Test
fig, ax  = create_actual_vs_expected_plot(
    X_sum, "claim_size", "pred_claims", 
    'Test QQ plot 20', 
    writer, 'QQ plot Test'
)

fig


In [ ]:

# Logged QQ - Test
fig, ax  = create_actual_vs_expected_plot(
    X_sum, "log_actual", "log_pred_claims", 
    'Logged Test QQ plot 20', 
    writer, 'QQ plot Logged Test', 
    max_val = 20,
)

fig

## Occurrence Period Plot

We can review the accuracy of the trained model by looking at Actuals and Expected values by Occrrence Period. 

In [ ]:

datTrainUlt=train_pred.groupby("claim_no").last()
datTestUlt=test_pred.groupby("claim_no").last()

datTrain_occ = datTrainUlt.groupby("occurrence_period").agg({youtput: "sum", "pred_claims": "sum"})
datTest_occ = datTestUlt.groupby("occurrence_period").agg({youtput: "sum", "pred_claims": "sum"})

plt.figure()

plt.plot(datTrain_occ.index, datTrain_occ[youtput])
plt.plot(datTrain_occ.index, datTrain_occ.pred_claims)

plt.plot(datTest_occ.index, datTest_occ[youtput])
plt.plot(datTest_occ.index, datTest_occ.pred_claims)

fig, ax = plt.subplots()
ax.plot(datTrain_occ.index, datTrain_occ[youtput], linestyle='--', label='Train Actual')
ax.plot(datTrain_occ.index, datTrain_occ.pred_claims, linestyle='--', label='Train Expected')
ax.plot(datTest_occ.index, datTest_occ[youtput], label='Test Actual')
ax.plot(datTest_occ.index, datTest_occ.pred_claims, label='Test Expected')
ax.set_yscale("log") 
ax.set_xlabel('Occurrence period', fontsize=15)
ax.set_ylabel('Total Ultimate claims', fontsize=15)
ax.set_title('by Occurrence Period')     
ax.legend()
writer.add_figure('by Occur Period', fig)


fig


# Development date plots


In [ ]:
datTrain_dev = train_pred.groupby("development_period").agg({youtput: "sum", "pred_claims": "sum"})
datTest_dev = test_pred.groupby("development_period").agg({youtput: "sum", "pred_claims": "sum"})

plt.figure()

plt.plot(datTrain_dev.index, datTrain_dev[youtput])
plt.plot(datTrain_dev.index, datTrain_dev.pred_claims)

plt.plot(datTest_dev.index, datTest_dev[youtput])
plt.plot(datTest_dev.index, datTest_dev.pred_claims)


fig, ax = plt.subplots()
ax.plot(datTrain_dev.index, datTrain_dev[youtput], linestyle='--', label='Train Actual')
ax.plot(datTrain_dev.index, datTrain_dev.pred_claims, linestyle='--', label='Train Expected')
ax.plot(datTest_dev.index, datTest_dev[youtput], label='Test Actual')
ax.plot(datTest_dev.index, datTest_dev.pred_claims, label='Test Expected')
ax.set_xlabel('Development period', fontsize=15)
ax.set_ylabel('Total claims', fontsize=15)
ax.set_title('by Devevelopment Period')     
ax.legend()
writer.add_figure('by Dev Period', fig)

fig

## Development Period Plot

We can review the accuracy of the trained model by looking at Actuals and Expected values by Development Period. 

In [ ]:
datTrain_dev = train_pred.groupby("development_period").agg({youtput: "sum", "pred_claims": "sum"})
datTest_dev = test_pred.groupby("development_period").agg({youtput: "sum", "pred_claims": "sum"})

plt.figure()

plt.plot(datTrain_dev.index, datTrain_dev[youtput])
plt.plot(datTrain_dev.index, datTrain_dev.pred_claims)

plt.plot(datTest_dev.index, datTest_dev[youtput])
plt.plot(datTest_dev.index, datTest_dev.pred_claims)


fig, ax = plt.subplots()
ax.plot(datTrain_dev.index, datTrain_dev[youtput], linestyle='--', label='Train Actual')
ax.plot(datTrain_dev.index, datTrain_dev.pred_claims, linestyle='--', label='Train Expected')
ax.plot(datTest_dev.index, datTest_dev[youtput], label='Test Actual')
ax.plot(datTest_dev.index, datTest_dev.pred_claims, label='Test Expected')
ax.set_xlabel('Development period', fontsize=15)
ax.set_ylabel('Total claims', fontsize=15)
ax.set_title('by Devevelopment Period')     
ax.legend()
writer.add_figure('by Dev Period', fig)

fig